# Лабораторная работа 1. Продвинутый NumPy
**Предмет:** Язык Python для анализа данных  
**Выполнил:** Михайлов Яков Витальевич  
**Группа:** 1521741

Тема 1. NumPy
Запускайте ячейки последовательно. Времена выполнения зависят от компьютера.
Для задания 5 нужен компилятор C; в GitHub Codespaces он доступен.


In [1]:
from pathlib import Path
from decimal import Decimal
import sys
import subprocess
import platform
import numpy as np
import scipy
from scipy import sparse

LAB_DIR = Path.cwd()
if not (LAB_DIR / "sum_squares.pyx").exists():
    LAB_DIR = LAB_DIR / "labs" / "01_numpy"
assert (LAB_DIR / "sum_squares.pyx").exists(), "Откройте блокнот внутри репозитория"
print("Python:", platform.python_version())
print("NumPy:", np.__version__, "| SciPy:", scipy.__version__)


Python: 3.12.14
NumPy: 2.3.5 | SciPy: 1.17.0


## 1. Числа с плавающей точкой
### 1.1. Сумма и сравнение с точным значением
Для точного десятичного эталона используем Decimal, созданный из строки.
Печать массива может скрывать погрешность округления.


In [2]:
exact = Decimal("0.6")
for dtype in (np.float32, np.float64):
    a = np.array([0.1, 0.2, 0.3], dtype=dtype)
    total = a.sum()
    represented = Decimal.from_float(float(total))
    print(dtype.__name__)
    print("Сумма:", represented)
    print("Абсолютная погрешность:", abs(represented - exact))
    print("Близка к 0.6:", np.isclose(total, 0.6))


float32
Сумма: 0.60000002384185791015625
Абсолютная погрешность: 2.384185791015625E-8
Близка к 0.6: True
float64
Сумма: 0.600000000000000088817841970012523233890533447265625
Абсолютная погрешность: 8.881784197001252323389053345E-17
Близка к 0.6: True


### 1.2. Машинное эпсилон
Это расстояние от 1 до следующего представимого числа данного типа.

In [3]:
for dtype in (np.float32, np.float64):
    eps = np.finfo(dtype).eps
    print(dtype.__name__, "eps =", eps)
    print("1 + eps > 1:", dtype(1) + eps > dtype(1))


float32 eps = 1.1920929e-07
1 + eps > 1: True
float64 eps = 2.220446049250313e-16
1 + eps > 1: True


### 1.3. Катастрофическая отмена
Разность исходных десятичных чисел 1.0000001 и 1.0000000 равна 0.0000001.
При вычитании близких приближённых значений относительная ошибка разности
может быть намного больше относительных ошибок операндов.


In [4]:
exact_diff = Decimal("1.0000001") - Decimal("1.0000000")
for dtype in (np.float32, np.float64):
    x, y = dtype("1.0000001"), dtype("1.0000000")
    diff = x - y
    actual = Decimal.from_float(float(diff))
    relative = abs(actual - exact_diff) / abs(exact_diff)
    print(dtype.__name__, "разность =", actual)
    print("Относительная ошибка:", float(relative), "или", float(relative * 100), "%")


float32 разность = 1.1920928955078125E-7
Относительная ошибка: 0.1920928955078125 или 19.20928955078125 %
float64 разность = 1.000000000583867176828789524734020233154296875E-7
Относительная ошибка: 5.838671768287895e-10 или 5.838671768287895e-08 %


**Вывод:** float64 точнее float32, но тоже имеет конечную точность.
При вычитании близких чисел ранее возникшее округление становится существенным
относительно малой разности; само вычитание не обязательно создаёт новую ошибку.


## 2. Индексация, срезы и broadcasting
### 2.1. Строка, столбец и подмассив
Индексация начинается с нуля. «Вторая строка» имеет индекс 1.
Фразу «строки 1–3 и столбцы 2–4» трактуем как индексы включительно;
правая граница среза не включается. Для порядковых номеров вариант приведён отдельно.


In [5]:
A = np.arange(-10, 10).reshape(4, 5)
print("A:\n", A)
print("Вторая строка:", A[1, :])
print("Столбец с индексом 3:", A[:, 3])
print("Индексы строк 1–3, столбцов 2–4:\n", A[1:4, 2:5])
print("Если имелись в виду порядковые номера 1–3 и 2–4:\n", A[0:3, 1:4])


A:
 [[-10  -9  -8  -7  -6]
 [ -5  -4  -3  -2  -1]
 [  0   1   2   3   4]
 [  5   6   7   8   9]]
Вторая строка: [-5 -4 -3 -2 -1]
Столбец с индексом 3: [-7 -2  3  8]
Индексы строк 1–3, столбцов 2–4:
 [[-3 -2 -1]
 [ 2  3  4]
 [ 7  8  9]]
Если имелись в виду порядковые номера 1–3 и 2–4:
 [[-9 -8 -7]
 [-4 -3 -2]
 [ 1  2  3]]


### 2.2. Fancy-индексация
np.ix_ выбирает все сочетания указанных строк и столбцов, получаем массив 3 × 2.

In [6]:
selected = A[np.ix_([0, 2, 3], [1, 4])]
print(selected)
assert selected.shape == (3, 2)


[[-9 -6]
 [ 1  4]
 [ 6  9]]


### 2.3. Логическая индексация

In [7]:
nonnegative = A.copy()
nonnegative[nonnegative < 0] = 0
print(nonnegative)
assert np.all(nonnegative >= 0)


[[0 0 0 0 0]
 [0 0 0 0 0]
 [0 1 2 3 4]
 [5 6 7 8 9]]


### 2.4. Broadcasting без циклов

In [8]:
B = np.arange(4).reshape(4, 1)
C = np.arange(5).reshape(1, 5)
result = B + C
print("B:\n", B, "\nC:\n", C)
print("B + C:\n", result)
assert result.shape == (4, 5)


B:
 [[0]
 [1]
 [2]
 [3]] 
C:
 [[0 1 2 3 4]]
B + C:
 [[0 1 2 3 4]
 [1 2 3 4 5]
 [2 3 4 5 6]
 [3 4 5 6 7]]


**Вывод:** размеры (4, 1) и (1, 5) совместимы; результат имеет форму (4, 5).

## 3. Memory-mapped файлы и порядок доступа
### 3.1. Создание файла
Массив 1000 × 1000 float64 занимает 8 000 000 байт (около 7.63 MiB).
Сам пример помещается в ОЗУ; он демонстрирует подход к более крупным данным.
Заполняем по строкам, не создавая второй полный массив.


In [9]:
data_dir = LAB_DIR / "data"
data_dir.mkdir(exist_ok=True)
file_path = data_dir / "matrix_float64.dat"
rng = np.random.default_rng(42)
mapped = np.memmap(file_path, dtype=np.float64, mode="w+", shape=(1000, 1000), order="C")
for i in range(mapped.shape[0]):
    mapped[i, :] = rng.random(mapped.shape[1])
mapped.flush()
del mapped
print("Размер файла:", file_path.stat().st_size, "байт")


Размер файла: 8000000 байт


### 3.2. Чтение отдельной строки и столбца
Срезы создают представления; не вызываем копирование всего массива в ОЗУ. ОС загружает нужные страницы файла.

In [10]:
mapped = np.memmap(file_path, dtype=np.float64, mode="r", shape=(1000, 1000), order="C")
row = mapped[123, :]
column = mapped[:, 456]
print("Строка:", row[:10])
print("Столбец:", column[:10])
print("Общая память с исходным массивом:", np.shares_memory(mapped, row), np.shares_memory(mapped, column))


Строка: [0.0839493  0.26373944 0.45679629 0.88020128 0.68295222 0.57217447
 0.75158842 0.58202074 0.08789948 0.58822181]
Столбец: [0.98002718 0.22592896 0.60619643 0.93756526 0.14897782 0.178246
 0.47203895 0.73116391 0.95219254 0.20457075]
Общая память с исходным массивом: True True


### 3.3. Сравнение с помощью %timeit
Для обоих способов выполняем 1000 суммирований по 1000 элементов.
Предварительно читаем данные: измеряем повторные обращения с прогретым файловым кэшем,
а не скорость первого чтения с диска. Это не измерение чистой пропускной способности кэша CPU:
в результат также входят вызовы Python и NumPy.


In [11]:
def by_rows(a):
    total = 0.0
    for i in range(a.shape[0]):
        total += a[i, :].sum()
    return total

def by_columns(a):
    total = 0.0
    for j in range(a.shape[1]):
        total += a[:, j].sum()
    return total

assert np.isclose(by_rows(mapped), by_columns(mapped))
print("Шаги в байтах:", mapped.strides)
rows_time = %timeit -o -r 5 -n 5 by_rows(mapped)
columns_time = %timeit -o -r 5 -n 5 by_columns(mapped)
print(f"Время столбцов / время строк: {columns_time.average / rows_time.average:.2f}")


Шаги в байтах: (8000, 8)
6.83 ms +- 366 us per loop (mean +- std. dev. of 5 runs, 5 loops each)
10.2 ms +- 975 us per loop (mean +- std. dev. of 5 runs, 5 loops each)
Время столбцов / время строк: 1.49


**Объяснение:** в C-порядке соседние элементы строки находятся рядом (шаг 8 байт),
а элементы столбца идут с шагом 8000 байт. Поэтому обход строк обычно эффективнее
использует кэш. На небольшом массиве разницу могут перекрывать накладные расходы,
нагрузка системы и файловый кэш; конкретный порядок результатов не гарантирован.


## 4. Разреженные матрицы
### 4.1–4.3. CSR, COO, CSC и умножение на вектор

In [12]:
rng_sparse = np.random.default_rng(42)
csr = sparse.random(5, 5, density=0.2, format="csr",
                    random_state=rng_sparse, data_rvs=rng_sparse.random)
coo = csr.tocoo()
csc = csr.tocsc()
vector = np.arange(1, 6, dtype=np.float64)
print("Матрица:\n", csr.toarray())
print("Форматы:", csr.format, coo.format, csc.format)
print("Ненулевых элементов:", csr.nnz, "| Плотность:", csr.nnz / 25)
print("Результат:", csr @ vector)
np.testing.assert_allclose(csr @ vector, csr.toarray() @ vector)
np.testing.assert_array_equal(coo.toarray(), csc.toarray())


Матрица:
 [[0.         0.         0.7611397  0.78606431 0.        ]
 [0.12811363 0.         0.         0.         0.        ]
 [0.         0.         0.         0.97562235 0.        ]
 [0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.45038594]]
Форматы: csr coo csc
Ненулевых элементов: 5 | Плотность: 0.2
Результат: [5.42767633 0.12811363 3.90248941 0.         2.25192969]


### 4.4. Сравнение при размере 1000 × 1000 и плотности 0.01
Плотную и разреженную версии строим из одной матрицы.
Создание и преобразование не входят в замер; результаты умножения проверяем.


In [13]:
large = sparse.random(1000, 1000, density=0.01, format="csr",
                      random_state=rng_sparse, data_rvs=rng_sparse.random)
dense = large.toarray()
v = rng_sparse.random(1000)
np.testing.assert_allclose(large @ v, dense @ v, rtol=1e-12, atol=1e-12)
sparse_time = %timeit -o -r 5 -n 20 large @ v
dense_time = %timeit -o -r 5 -n 20 dense @ v
csr_bytes = large.data.nbytes + large.indices.nbytes + large.indptr.nbytes
print("Память CSR:", csr_bytes, "байт; плотной:", dense.nbytes, "байт")
print(f"Время dense / время CSR: {dense_time.average / sparse_time.average:.2f}")


24 us +- 3.73 us per loop (mean +- std. dev. of 5 runs, 20 loops each)
The slowest run took 4.17 times longer than the fastest. This could mean that an intermediate result is being cached.
5.6 ms +- 2.82 ms per loop (mean +- std. dev. of 5 runs, 20 loops each)
Память CSR: 124004 байт; плотной: 8000000 байт
Время dense / время CSR: 233.35


**Вывод:** CSR хранит значения, индексы столбцов и границы строк; COO — значения
и пары координат, CSC — сжатое представление по столбцам.
При низкой плотности CSR экономит память и обычно ускоряет произведение на вектор.
Скорость также зависит от BLAS, потоков и оборудования.


## 5. Нативное расширение Cython
### 5.1. Python-функция для списка


In [14]:
def sum_squares_python(values):
    total = 0.0
    for value in values:
        total += value * value
    return total

assert sum_squares_python([1.0, 2.0, 3.0]) == 14.0


### 5.2. Компиляция отдельного файла .pyx
В файле sum_squares.pyx реализована функция с типизированными переменными
и доступом к непрерывному массиву float64 через memoryview.
Компиляция выполняется отдельно от измерения времени.


In [15]:
build = subprocess.run(
    [sys.executable, "setup.py", "build_ext", "--inplace"],
    cwd=LAB_DIR, capture_output=True, text=True
)
print(build.stdout)
if build.returncode:
    print(build.stderr)
build.check_returncode()
sys.path.insert(0, str(LAB_DIR))
from sum_squares import sum_squares_cython
assert sum_squares_cython(np.array([1.0, 2.0, 3.0])) == 14.0


running build_ext
building 'sum_squares' extension
gcc -fno-strict-overflow -Wsign-compare -Wunreachable-code -DNDEBUG -g -O3 -Wall -O3 -fstack-protector -fstack-clash-protection -U_FORTIFY_SOURCE -D_FORTIFY_SOURCE=2 -fno-omit-frame-pointer -mno-omit-leaf-frame-pointer -fPIC -fPIC -I/opt/codex/runtimes/codex-primary-runtime/dependencies/python/include/python3.12 -c /workspace/scratch/6501af650f8f/python-data-analysis-mikhailov/labs/01_numpy/sum_squares.c -o build/temp.linux-x86_64-cpython-312/workspace/scratch/6501af650f8f/python-data-analysis-mikhailov/labs/01_numpy/sum_squares.o
gcc -shared build/temp.linux-x86_64-cpython-312/workspace/scratch/6501af650f8f/python-data-analysis-mikhailov/labs/01_numpy/sum_squares.o -L/install/lib -o build/lib.linux-x86_64-cpython-312/sum_squares.cpython-312-x86_64-linux-gnu.so
copying build/lib.linux-x86_64-cpython-312/sum_squares.cpython-312-x86_64-linux-gnu.so -> 



### 5.3. Замер на 10⁷ элементах
Создаём массив float64 и список с теми же значениями. Python-функцию проверяем
на списке и на массиве; Cython — на том же массиве. Это позволяет отдельно
увидеть влияние представления входных данных.
Список и массив вместе требуют несколько сотен мегабайт ОЗУ.
Создание данных, преобразование в список и компиляция не входят в замеры.


In [16]:
N = 10**7
values = np.random.default_rng(42).random(N)
values_list = values.tolist()
expected = np.sum(values * values)
np.testing.assert_allclose(sum_squares_python(values_list), expected, rtol=1e-10)
np.testing.assert_allclose(sum_squares_cython(values), expected, rtol=1e-10)
print("Число элементов:", N)
python_list_time = %timeit -o -r 3 -n 1 sum_squares_python(values_list)
python_array_time = %timeit -o -r 3 -n 1 sum_squares_python(values)
cython_time = %timeit -o -r 3 -n 1 sum_squares_cython(values)
print(f"Ускорение относительно Python на списке: {python_list_time.average / cython_time.average:.2f}")
print(f"Ускорение относительно Python на массиве: {python_array_time.average / cython_time.average:.2f}")
del values_list, values


Число элементов: 10000000
707 ms +- 444 ms per loop (mean +- std. dev. of 3 runs, 1 loop each)
3.74 s +- 1.41 s per loop (mean +- std. dev. of 3 runs, 1 loop each)
39.3 ms +- 3.75 ms per loop (mean +- std. dev. of 3 runs, 1 loop each)
Ускорение относительно Python на списке: 17.99
Ускорение относительно Python на массиве: 95.15


**Вывод:** Cython переводит типизированный цикл в скомпилированный код,
уменьшая расходы на интерпретацию и Python-объекты.
Входные данные и результаты совпадают в пределах погрешности float64.
Численные значения времени являются результатами конкретного запуска, а не гарантией ускорения.
